In [9]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier

# ==========================================
# 1. CONFIGURATION
# ==========================================
SYMBOL = 'IRCON.NS'       # Ircon International Ltd
START_DATE = '2023-01-01'
END_DATE = '2026-01-14'   # Covers Jan 1 - Jan 8, 2026
STOP_LOSS_PCT = 0.01      # 1% Stop Loss
TAKE_PROFIT_PCT = 0.02    # 2% Take Profit

# ==========================================
# 2. DATA LOADING & FEATURE ENGINEERING
# ==========================================
def fetch_data(symbol):
    print(f"Downloading data for {symbol}...")
    df = yf.download(symbol, start=START_DATE, end=END_DATE, progress=False)

    # Handle multi-index columns (fix for recent yfinance versions)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    # Calculate Daily Returns
    df['Returns'] = df['Close'].pct_change()

    # --- Feature Generation ---
    # 1. Trend (Simple Moving Averages)
    df['SMA_10'] = df['Close'].rolling(window=10).mean()
    df['SMA_50'] = df['Close'].rolling(window=50).mean()

    # 2. Momentum (RSI)
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))

    # 3. Volatility
    df['Volatility'] = df['Returns'].rolling(window=10).std()

    # 4. Lagged Returns (Autoregression)
    df['Lag_1'] = df['Returns'].shift(1)
    df['Lag_2'] = df['Returns'].shift(2)

    # --- Target Variable ---
    # 1 = Bullish (Next Close > Today's Close), 0 = Bearish
    df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

    df.dropna(inplace=True)
    return df

# ==========================================
# 3. CHRONOLOGICAL WALK-FORWARD TRAINING
# ==========================================
def run_walk_forward_model(df):
    print("Training Model with Walk-Forward Validation...")

    model = RandomForestClassifier(n_estimators=100, min_samples_split=10, random_state=42)
    features = ['SMA_10', 'SMA_50', 'RSI', 'Volatility', 'Lag_1', 'Lag_2']

    predictions = []

    # Initial Training Window (e.g., first 200 days)
    start_index = 200

    # Iterate through time, day by day
    for i in range(start_index, len(df)):
        # Train on PAST data only (0 to i)
        train_data = df.iloc[:i]
        # Predict the NEXT day (i)
        test_row = df.iloc[i:i+1]

        # Periodic Re-training (e.g., every 30 days) to keep model fresh
        if (i - start_index) % 30 == 0:
            model.fit(train_data[features], train_data['Target'])

        pred = model.predict(test_row[features])[0]
        predictions.append(pred)

    # Align predictions with dates
    results = df.iloc[start_index:].copy()
    results['Prediction'] = predictions
    return results

# ==========================================
# 4. STRATEGY REPORTING (JAN 1 - JAN 8)
# ==========================================
def generate_strategy_report(df):
    # Filter for the specific contest window
    target_window = df.loc['2026-01-01':'2026-01-08'].copy()

    if target_window.empty:
        print("No data found for the requested period (Jan 1-8, 2026).")
        return

    # We need the previous day's prediction to determine the "Action"
    start_pos = df.index.get_loc(target_window.index[0])
    # Extract prediction sequence covering the window + 1 day prior
    prediction_sequence = df['Prediction'].iloc[start_pos-1 : start_pos + len(target_window)].values

    print("\n" + "="*75)
    print(f"TRADING STRATEGY REPORT: {SYMBOL} (JAN 1 - JAN 8, 2026)")
    print("="*75)
    print(f"{'Date':<12} | {'Close Price':<12} | {'ML Pred':<8} | {'Strategy Action':<30}")
    print("-" * 75)

    for i in range(len(target_window)):
        date = target_window.index[i]
        close = target_window['Close'].iloc[i]

        curr_pred = prediction_sequence[i+1] # Prediction for tomorrow
        prev_pred = prediction_sequence[i]   # Existing state from yesterday

        # --- Logic Differentiation ---
        # 1 = Bullish Signal, 0 = Bearish Signal

        if curr_pred == 1 and prev_pred == 0:
            action = "BUY (Entry Signal)"
        elif curr_pred == 1 and prev_pred == 1:
            action = "HOLD (Maintain Long)"
        elif curr_pred == 0 and prev_pred == 1:
            action = "SELL (Exit Signal)"
        elif curr_pred == 0 and prev_pred == 0:
            action = "AVOID (Stay Flat/Cash)"

        print(f"{date.strftime('%Y-%m-%d'):<12} | {close:<12.2f} | {curr_pred:<8} | {action:<30}")

    print("-" * 75)
    print("RISK MANAGEMENT RULES (Applied during execution):")
    print(f"* Stop Loss:   {STOP_LOSS_PCT*100}% movement against entry.")
    print(f"* Take Profit: {TAKE_PROFIT_PCT*100}% profit secured.")
    print("="*75)

# ==========================================
# 5. EXECUTION
# ==========================================
# 1. Fetch
data = fetch_data(SYMBOL)

# 2. Train & Predict
backtest_results = run_walk_forward_model(data)

# 3. Generate Report
generate_strategy_report(backtest_results)

Training Model with Walk-Forward Validation...

TRADING STRATEGY REPORT: IRCON.NS (JAN 1 - JAN 8, 2026)
Date         | Close Price  | ML Pred  | Strategy Action               
---------------------------------------------------------------------------
2026-01-01   | 177.98       | 1        | HOLD (Maintain Long)          
2026-01-02   | 179.22       | 1        | HOLD (Maintain Long)          
2026-01-05   | 177.02       | 1        | HOLD (Maintain Long)          
2026-01-06   | 177.26       | 0        | SELL (Exit Signal)            
2026-01-07   | 177.42       | 0        | AVOID (Stay Flat/Cash)        
2026-01-08   | 169.78       | 1        | BUY (Entry Signal)            
---------------------------------------------------------------------------
RISK MANAGEMENT RULES (Applied during execution):
* Stop Loss:   1.0% movement against entry.
* Take Profit: 2.0% profit secured.
